# Meteorological Data — Preprocessing & Feature Engineering
This notebook loads `Dataset/meteo-historical-data.json` and demonstrates robust preprocessing and feature engineering steps suitable for time-series meteorological data.

Cells: 1) imports & loading, 2) inspection, 3) cleaning/missing values, 4) outliers, 5) feature engineering, 6) scaling/saving.
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "**Modeling Overview — EDA, seasonal modelling, training & comparison**",
                "\n",
                "This section performs: 1) exploratory data analysis focused on rainfall and seasonal patterns,",
                "2) rainfall event identification and characterization, 3) model training and tuning for Logistic Regression, SVM, and Random Forest using time-aware CV, and 4) performance comparison and model persistence."
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# Modeling prerequisites: imports and helpers",
                "import matplotlib.pyplot as plt",
                "import seaborn as sns",
                "from sklearn.linear_model import LogisticRegression",
                "from sklearn.svm import SVC",
                "from sklearn.ensemble import RandomForestClassifier",
                "from sklearn.model_selection import GridSearchCV, TimeSeriesSplit",
                "from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,",
                "                             confusion_matrix, roc_curve, auc)",
                "import pickle",
                "",
                "plt.rcParams.update({'figure.max_open_warning': 0})",
                "print('Modeling imports ready')"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 1. Exploratory Data Analysis",
                "# Identify the rainfall column if not already set",
                "if 'rain_col' not in globals() or rain_col is None:",
                "    rain_col = None",
                "    for c in df.columns:",
                "        if any(k in c.lower() for k in ['pr','rain','precip']):",
                "            rain_col = c",
                "            break",
                "if rain_col is None:",
                "    raise RuntimeError('Rainfall column not found; adjust detection logic or set `rain_col` manually')",
                "",
                "# 1.1 Descriptive statistics",
                "display(df.describe(include='all').T)",
                "",
                "# 1.2 Seasonal rainfall pattern: mean and median per season/month",
                "season_mean = df.groupby('season')[rain_col].mean().reindex(['DJF','MAM','JJA','SON'])",
                "season_median = df.groupby('season')[rain_col].median().reindex(['DJF','MAM','JJA','SON'])",
                "print('Seasonal mean rainfall:\n', season_mean)",
                "print('\nSeasonal median rainfall:\n', season_median)",
                "",
                "# Plot seasonal means",
                "fig, ax = plt.subplots(1,2,figsize=(12,4))",
                "season_mean.plot(kind='bar', ax=ax[0], title='Mean rainfall by season')",
                "season_median.plot(kind='bar', ax=ax[1], title='Median rainfall by season')",
                "plt.tight_layout()",
                "",
                "# 1.3 Temporal pattern by month (use month numeric)",
                "monthly = df.groupby('month')[rain_col].agg(['mean','median','count'])",
                "display(monthly)",
                "plt.figure(figsize=(10,4))",
                "monthly['mean'].plot(title='Mean monthly rainfall')",
                "plt.xlabel('Month')",
                "plt.ylabel(rain_col)",
                "plt.show()"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 2. Rainfall event identification and characterization",
                "# Define a rainfall event threshold (mm). Adjust per thesis if specified.",
                "threshold = 1.0",
                "df['rain_event'] = (df[rain_col] > threshold).astype(int)",
                "print('Rain event threshold:', threshold)",
                "print('Event distribution:')",
                "display(df['rain_event'].value_counts())",
                "",
                "# Identify contiguous events (runs of rain_event==1)",
                "df['event_change'] = (df['rain_event'] != df['rain_event'].shift(1)).cumsum()",
                "events = df[df['rain_event']==1].groupby('event_change').agg(",
                "    start_time=('rain_event', lambda x: x.index[0]),",
                "    end_time=('rain_event', lambda x: x.index[-1]),",
                "    duration=('rain_event', 'size'),",
                "    total_rain=(rain_col, 'sum')",
                ")",
                "# normalize index if any",
                "events = events.reset_index(drop=True)",
                "display(events.sort_values('total_rain', ascending=False).head(10))",
                "",
                "# cleanup helper column",
                "df = df.drop(columns=['event_change'])"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 3. Prepare data for modeling",
                "# Use engineered features created earlier (lag_features, temp_range, season, month, year, is_weekend)",
                "features = []",
                "features += [f for f in (lag_features if 'lag_features' in globals() else []) if f in df.columns]",
                "if 'temp_range' in df.columns:",
                "    features.append('temp_range')",
                "for c in ['is_weekend','month','year','dayofweek']:",
                "    if c in df.columns and c not in features:",
                "        features.append(c)",
                "# One-hot encode season if present",
                "if 'season' in df.columns:",
                "    season_dummies = pd.get_dummies(df['season'], prefix='season', drop_first=True)",
                "    df = pd.concat([df, season_dummies], axis=1)",
                "    features += list(season_dummies.columns)",
                "",
                "# Final features and target",
                "X = df[features].copy()",
                "y = df['rain_event'].copy()",
                "# Drop rows with NaNs in features/target",
                "mask = X.notna().all(axis=1) & y.notna()",
                "X = X[mask].copy(); y = y[mask].copy()",
                "print('Final feature shape:', X.shape, 'Target shape:', y.shape)",
                "display(X.head())"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 4. Train/test split (time-aware): hold out last 20% as test set",
                "split_idx = int(len(X) * 0.8)",
                "X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]",
                "y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]",
                "print('Train size:', X_train.shape, 'Test size:', X_test.shape)",
                "",
                "# Use TimeSeriesSplit for CV during hyperparameter search",
                "tscv = TimeSeriesSplit(n_splits=5)",
                "print('TimeSeriesSplit configured with n_splits=5')"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 5. Model definitions and hyperparameter grids (adjust to thesis hyperparams if provided)",
                "models = {",
                "    'LogisticRegression': (LogisticRegression(solver='liblinear', max_iter=1000),",
                "        {'C':[0.01,0.1,1,10], 'penalty':['l2']}),",
                "    'SVM': (SVC(probability=True),",
                "        {'kernel':['rbf','linear'], 'C':[0.1,1,10], 'gamma':['scale','auto']}),",
                "    'RandomForest': (RandomForestClassifier(random_state=42),",
                "        {'n_estimators':[100,200], 'max_depth':[None,10,20], 'min_samples_split':[2,5]})",
                "}",
                "",
                "results = {}",
                "for name, (estimator, grid) in models.items():",
                "    print(f'Grid search for {name}...')",
                "    gs = GridSearchCV(estimator, grid, cv=tscv, scoring='roc_auc', n_jobs=-1, verbose=0)",
                "    gs.fit(X_train, y_train)",
                "    print(name, 'best params:', gs.best_params_, 'best CV AUC:', gs.best_score_)",
                "    results[name] = gs",
                "    # save best estimator",
                "    os.makedirs('artifacts', exist_ok=True)",
                "    with open(os.path.join('artifacts', f'{name}_best.pkl'),'wb') as f:",
                "        pickle.dump(gs.best_estimator_, f)",
                "    print('Saved', name, 'to artifacts/')"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 6. Evaluation on held-out test set",
                "eval_rows = []",
                "import numpy as _np",
                "for name, gs in results.items():",
                "    model = gs.best_estimator_",
                "    y_pred = model.predict(X_test)",
                "    y_proba = model.predict_proba(X_test)[:,1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)",
                "    acc = accuracy_score(y_test, y_pred)",
                "    prec = precision_score(y_test, y_pred, zero_division=0)",
                "    rec = recall_score(y_test, y_pred, zero_division=0)",
                "    f1 = f1_score(y_test, y_pred, zero_division=0)",
                "    try:",
                "        roc = roc_auc_score(y_test, y_proba)",
                "    except Exception:",
                "        roc = _np.nan",
                "    eval_rows.append((name, acc, prec, rec, f1, roc))",
                "    print('\\nModel:', name)",
                "    print('Accuracy: %.3f | Precision: %.3f | Recall: %.3f | F1: %.3f | ROC AUC: %.3f' % (acc, prec, rec, f1, roc if not _np.isnan(roc) else -1))",
                "    # confusion matrix",
                "    cm = confusion_matrix(y_test, y_pred)",
                "    print('Confusion matrix:\\n', cm)",
                "    # ROC curve",
                "    try:",
                "        fpr, tpr, _ = roc_curve(y_test, y_proba)",
                "        plt.figure(figsize=(6,4))",
                "        plt.plot(fpr, tpr, label=f'{name} (AUC={roc:.3f})')",
                "        plt.plot([0,1],[0,1],'k--')",
                "        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f'ROC - {name}'); plt.legend(); plt.show()",
                "    except Exception:",
                "        pass",
                "",
                "# Summary table",
                "import pandas as _pd",
                "summary = _pd.DataFrame(eval_rows, columns=['model','accuracy','precision','recall','f1','roc_auc']).set_index('model')",
                "display(summary)",
                "# persist summary",
                "summary.to_csv(os.path.join('artifacts','model_performance_summary.csv'))",
                "print('Saved performance summary to artifacts/model_performance_summary.csv')"
            ]
        },
<VSCode.Cell id="#VSC-780f2699" language="python">
# 1. Imports and settings
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [35]:
# 2. Load dataset (robust to common issues)
data_path = os.path.join('Dataset', 'meteo-historical-data.json')
if not os.path.exists(data_path):
    raise FileNotFoundError(f'Expected dataset at {data_path} — check path')

# Attempt to read as JSON lines or normal JSON
try:
    df = pd.read_json(data_path)
except ValueError:
    # try json lines
    df = pd.read_json(data_path, lines=True)

print('Loaded dataset with shape:', df.shape)
df.head()

Loaded dataset with shape: (18720, 1)


,observation_grided_data
0,"{'observe_grid10_id': 2341, 'observe_grid10_la..."
1,"{'observe_grid10_id': 2342, 'observe_grid10_la..."
2,"{'observe_grid10_id': 2343, 'observe_grid10_la..."
3,"{'observe_grid10_id': 2344, 'observe_grid10_la..."
4,"{'observe_grid10_id': 2345, 'observe_grid10_la..."


In [36]:
# 3. Quick inspection: columns, dtypes, missing values
display(df.info())
display(df.describe(include='all').T)
missing = df.isna().mean().sort_values(ascending=False)
display(missing[missing>0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18720 entries, 0 to 18719
Data columns (total 1 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   observation_grided_data  18720 non-null  object
dtypes: object(1)
memory usage: 146.4+ KB


None

,count,unique,top,freq
observation_grided_data,18720,18720,"{'observe_grid10_id': 2341, 'observe_grid10_la...",1


Series([], dtype: float64)

In [37]:
# 4. Robust datetime parsing: detect common time columns and parse
time_cols = [c for c in df.columns if c.lower() in ('date','datetime','time','timestamp','observed_at','ts')]
if len(time_cols) == 0:
    # try to infer any column with 'date' or time-like strings
    for c in df.columns:
        sample = df[c].dropna().astype(str).head(20).tolist()
        if any(('T' in s and ':' in s) or ('-' in s and ':' in s) for s in sample):
            time_cols.append(c)
            break

time_col = time_cols[0] if len(time_cols) else None
if time_col is None:
    print('No obvious datetime column found — you may need to create one.')
else:
    print('Using time column:', time_col)
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce', infer_datetime_format=True)
    # drop rows where time couldn't be parsed
    n_before = len(df)
    df = df[~df[time_col].isna()].copy()
    print(f'Dropped {n_before - len(df)} rows with unparsable timestamps')
    df = df.sort_values(time_col).reset_index(drop=True)
    df = df.set_index(time_col)
    # ensure a regular frequency if possible (optional)
    try:
        inferred = pd.infer_freq(df.index[:20])
        print('Inferred frequency (first 20 rows):', inferred)
    except Exception:
        pass

Using time column: observation_grided_data
Dropped 18720 rows with unparsable timestamps


In [38]:
# 5. Clean duplicates and low-information columns
n_dup = df.reset_index().duplicated().sum()
print('Duplicate rows (including timestamp):', n_dup)
if n_dup>0:
    df = df[~df.reset_index().duplicated()].copy()

# Drop columns with >50% missing values (tunable)
missing_frac = df.isna().mean()
drop_cols = missing_frac[missing_frac > 0.5].index.tolist()
if drop_cols:
    print('Dropping high-missing columns:', drop_cols)
    df = df.drop(columns=drop_cols)

# Optionally convert obvious numeric columns that are object dtype
for c in df.select_dtypes(include=['object']).columns:
    # skip index name if present
    try:
        df[c] = pd.to_numeric(df[c], errors='ignore')
    except Exception:
        pass

df.shape

Duplicate rows (including timestamp): 0


(0, 0)

In [39]:
# 6. Missing value strategies (numeric vs categorical)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
print('Numeric cols:', num_cols)
print('Categorical cols:', cat_cols)

# Numeric: prefer interpolation for time series, fallback to median
for c in num_cols:
    if df[c].isna().any():
        try:
            df[c] = df[c].interpolate(method='time')
        except Exception:
            df[c] = df[c].fillna(df[c].median())

# Categorical: fill with mode
for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode().iloc[0] if not df[c].mode().empty else 'missing')

# If still any missing remain, forward-fill then back-fill as a last resort
if df.isna().sum().sum() > 0:
    df = df.ffill().bfill()

display(df.isna().sum()[df.isna().sum()>0])

Numeric cols: []
Categorical cols: []


Series([], dtype: float64)

In [40]:
# 7. Outlier handling using IQR capping (robust)
# Protect against an empty/incorrectly loaded DataFrame
if df.empty or len(df.columns) == 0:
    print("DataFrame is empty or has no columns; attempting to normalize the raw JSON payload.")
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, dict):
            record_lists = [
                (key, value)
                for key, value in payload.items()
                if isinstance(value, list) and value and all(isinstance(item, dict) for item in value[:5])
            ]
            if not record_lists:
                raise ValueError("No list-of-dicts records found in JSON payload.")
            _, records = record_lists[0]
            df = pd.DataFrame(records)
        elif isinstance(payload, list):
            df = pd.DataFrame(payload)
        else:
            raise TypeError(f"Unsupported JSON structure: {type(payload).__name__}")
    except Exception as e:
        raise RuntimeError(f"Failed to normalize the dataset from {data_path}: {e}") from e

if df.empty or len(df.columns) == 0:
    raise ValueError("No usable columns remain after loading the dataset. Check the input JSON structure.")

# Recompute numeric columns after normalization
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

def cap_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower, upper)

for col in num_cols:
    if df[col].nunique() > 10:  # avoid capping boolean-like fields
        df[col] = cap_iqr(df[col])

# quick check after capping
if len(num_cols) > 0:
    display(df[num_cols].describe().T)
else:
    print("No numeric columns found; skipping IQR capping and summary.")

DataFrame is empty or has no columns; attempting to normalize the raw JSON payload.


,count,mean,std,min,25%,50%,75%,max
observe_grid10_id,18720.0,18790.700000,9092.711257,2341.000,11232.750,17784.50,26208.25,36972.000
observe_grid10_lat,18720.0,-1.705000,0.182967,-2.050,-1.850,-1.75,-1.55,-1.350
observe_grid10_lon,18720.0,29.565000,0.211607,29.150,29.425,29.55,29.75,29.950
observe_grid10_year,18720.0,2002.000000,11.254929,1983.000,1992.000,2002.00,2012.00,2021.000
observe_grid10_month,18720.0,6.500000,3.452145,1.000,3.750,6.50,9.25,12.000
observe_grid10_tas,18720.0,17.733056,2.409017,10.670,16.070,17.71,19.67,23.350
observe_grid10_tasmin,18720.0,12.587783,2.177618,6.665,11.180,12.49,14.19,18.705
observe_grid10_tasmax,18720.0,22.878361,2.980831,13.725,20.670,23.09,25.30,30.120
observe_grid10_pr,18720.0,119.841533,78.955475,0.000,60.000,116.00,171.00,337.500


In [41]:
# 8. Feature engineering (limited to features in thesis section 3.4)
# Ensure we have a datetime index
if not isinstance(df.index, pd.DatetimeIndex):
    if {'observe_grid10_year', 'observe_grid10_month'}.issubset(df.columns):
        df['date'] = pd.to_datetime(
            df['observe_grid10_year'].astype(str) + '-' +
            df['observe_grid10_month'].astype(str).str.zfill(2) + '-01',
            errors='coerce'
        )
        df = df.dropna(subset=['date']).sort_values('date').set_index('date')
    else:
        time_candidates = [c for c in df.columns if c.lower() in ('date','datetime','time','timestamp','observed_at','ts')]
        if time_candidates:
            time_col = time_candidates[0]
            df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
            df = df.dropna(subset=[time_col]).sort_values(time_col).set_index(time_col)
        else:
            raise RuntimeError('Index is not a DatetimeIndex; please set the parsed time column as index first.')

# Temporal components (as described in thesis)
df['year'] = df.index.year
df['month'] = df.index.month
# 'day' feature removed per request
df['dayofweek'] = df.index.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)

# Seasonal category (map months to seasons)
def month_to_season(m):
    if m in (12, 1, 2):
        return 'DJF'
    if m in (3, 4, 5):
        return 'MAM'
    if m in (6, 7, 8):
        return 'JJA'
    return 'SON'

df['season'] = df.index.month.map(month_to_season)

# Temperature range as in thesis: TempRange = tasmax - tasmin
tas_max_col = None
tas_min_col = None
for c in df.columns:
    cl = c.lower()
    if 'tasmax' in cl or 'tmax' in cl or 'temp_max' in cl:
        tas_max_col = c
    if 'tasmin' in cl or 'tmin' in cl or 'temp_min' in cl:
        tas_min_col = c

if tas_max_col and tas_min_col:
    df['temp_range'] = df[tas_max_col] - df[tas_min_col]

# Rainfall lag features t-1, t-2, t-3 (as described in thesis)
rain_col = None
for candidate in df.columns:
    if any(k in candidate.lower() for k in ['pr', 'rain', 'precip']):
        rain_col = candidate
        break

lag_features = []
if rain_col is not None:
    df['rainfall_t1'] = df[rain_col].shift(1)
    df['rainfall_t2'] = df[rain_col].shift(2)
    df['rainfall_t3'] = df[rain_col].shift(3)
    lag_features = ['rainfall_t1', 'rainfall_t2', 'rainfall_t3']

# Remove any additional engineered features not in the thesis: rolling stats, interactions, wind components
# (they are simply not created here)

# Drop rows with NaNs introduced by required lag features (if any)
if lag_features:
    df = df.dropna(subset=lag_features).copy()

print('Feature engineering (thesis-limited) completed. New shape:', df.shape)

Feature engineering (thesis-limited) completed. New shape: (18717, 19)


In [42]:
# 9. Encoding categorical variables
from sklearn.preprocessing import OneHotEncoder
cat_small = [c for c in cat_cols if df[c].nunique() < 20] if 'cat_cols' in locals() else []
# One-hot encode small categorical columns
if cat_small:
    df = pd.get_dummies(df, columns=cat_small, drop_first=True)

# Label encode remaining categoricals (if any)
label_encoders = {}
for c in df.select_dtypes(include=['object','category']).columns:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    label_encoders[c] = le

print('Encoding completed. Columns now:', df.shape[1])

Encoding completed. Columns now: 19


In [43]:
# 10. Scaling numeric features (fit scalers, save them)
num_final = df.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
df[num_final] = scaler.fit_transform(df[num_final])
# Persist scaler for later use in modeling pipelines
os.makedirs('artifacts', exist_ok=True)
with open(os.path.join('artifacts','scaler.pkl'),'wb') as f:
    pickle.dump(scaler, f)

print('Scaling complete and scaler saved to artifacts/scaler.pkl')

Scaling complete and scaler saved to artifacts/scaler.pkl


In [44]:
# 11. Save processed dataset for modeling (JSON)
out_path = os.path.join('Dataset', 'processed_meteo_data.json')
# Ensure 'day' is not included in the saved output per user request
if 'day' in df.columns:
    df = df.drop(columns=['day'])
# Convert to records and write a readable JSON file; dates are serialized via default=str
records = df.reset_index().to_dict(orient='records')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, default=str, indent=2)
print('Processed data saved to', out_path)

# Save label encoders if any
if label_encoders:
    with open(os.path.join('artifacts','label_encoders.pkl'),'wb') as f:
        pickle.dump(label_encoders, f)
    print('Saved label encoders to artifacts/label_encoders.pkl')

Processed data saved to Dataset/processed_meteo_data.json
Saved label encoders to artifacts/label_encoders.pkl


**Summary & Next steps**
- The dataset was loaded, parsed for datetime, cleaned for missing values, and had outliers capped.
- Temporal, lag, rolling, interaction, and wind-component features were created.
- Numeric features were scaled and artifacts (scalers/encoders) saved for model pipelines.

Next steps: exploratory visualization, feature selection, and building forecasting/classification models depending on the target.